# Demonstração: pedestre com cadeira de rodas

O vídeo da Cidade do México (notebook 07) não tem ninguém com auxílio de mobilidade. Aqui o sistema roda em dois
vídeos com uma pessoa em cadeira de rodas e mostra, quadro a quadro, a etiqueta **"Pedestre · cadeira de rodas"**.

**Os vídeos não têm faixa e a câmera se move** (acompanha as pessoas), então **não medem travessia**: servem só para
mostrar a identificação. Sem faixa, toda pessoa da cena é tratada como pedestre.

**Dois modelos juntos:**
- **YOLO-World** (sem treino): pessoas, veículos e auxílios, com rastreamento (ByteTrack);
- **YOLO11s ajustado** (notebook 08, treinado no Colab): pessoa **com** cadeira de rodas numa caixa só. Entra como
  detecção do auxílio; a associação (maior sobreposição) escolhe quem está sentado, não o acompanhante.

**Privacidade (LGPD):** além da cabeça de cada pessoa, o borrão usa o **rosto localizado por um modelo de pose**
(nariz, olhos, orelhas) e as pessoas encontradas pelos dois modelos. O bloco da pixelização cresce com o tamanho da
pessoa (em close, um bloco fixo deixaria o rosto reconhecível).

**Agrupamento na decisão:** a etiqueta diz o auxílio visto; o grupo de necessidade decide a ação na faixa
(cadeira, andador, muleta → mais tempo; bengala → mais tempo e sinal sonoro).

**Fontes:** Pexels 6646677 e 6646724 (licença Pexels, uso livre); ver `data/raw/video/FONTE.md`.

In [1]:
from pathlib import Path
import sys, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="torch")
import numpy as np, pandas as pd, cv2
ROOT = Path.cwd()
while not (ROOT / "src" / "visao").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src" / "visao")]
import visao as vs, detector
EVID, PROC, MODELS = ROOT / "docs" / "evidencias" / "estudo_b", ROOT / "data" / "processed", ROOT / "models"
VIDEOS = {"cadeirante_pexels_6646677": "de frente e de lado", "cadeirante_pexels_6646724": "de costas, em close"}
print("limiar do auxílio no YOLO-World:", vs.CONF_MIN_AUXILIO, "| limiar do modelo ajustado:", vs.CONF_AJUSTADO)

limiar do auxílio no YOLO-World: 0.1 | limiar do modelo ajustado: 0.25


## 1. Detecção (com cache)

Por quadro: YOLO-World com rastreamento, o modelo ajustado e o modelo de pose. O cache guarda **só números**
(caixas, classes, confianças), nunca imagem.

In [2]:
from ultralytics import YOLO
def detectar_tudo(vid):
    cache = PROC / f"visao_deteccoes_{vid}.parquet"
    if cache.exists():
        return pd.read_parquet(cache)
    ft, pose = YOLO(str(MODELS / "pcd_yolo11s.pt")), YOLO(str(MODELS / "yolo11n-pose.pt"))
    linhas, t0 = [], time.time()
    for i, img, dets in detector.detectar(ROOT / "data" / "raw" / "video" / f"{vid}.mp4"):
        linhas += [dict(quadro=i, fonte="yolo_world", classe=d.classe, x1=d.caixa[0], y1=d.caixa[1], x2=d.caixa[2],
                        y2=d.caixa[3], conf=d.conf, track_id=d.track_id) for d in dets]
        r = ft.predict(img, conf=0.1, verbose=False)[0]
        linhas += [dict(quadro=i, fonte="ajustado", classe=r.names[c], x1=b[0], y1=b[1], x2=b[2], y2=b[3], conf=p,
                        track_id=None)
                   for b, c, p in zip(r.boxes.xyxy.tolist(), r.boxes.cls.int().tolist(), r.boxes.conf.tolist())]
        r = pose.predict(img, conf=0.1, verbose=False)[0]
        if r.keypoints is not None and r.keypoints.conf is not None:
            for f in vs.rostos_pela_pose(r.keypoints.xy.cpu().numpy(), r.keypoints.conf.cpu().numpy(),
                                         r.boxes.xyxy.tolist(), i):
                linhas.append(dict(quadro=i, fonte="pose", classe="face", x1=f.caixa[0], y1=f.caixa[1],
                                   x2=f.caixa[2], y2=f.caixa[3], conf=1.0, track_id=None))
        linhas += [dict(quadro=i, fonte="pose", classe="person", x1=b[0], y1=b[1], x2=b[2], y2=b[3], conf=1.0,
                        track_id=None) for b in r.boxes.xyxy.tolist()]
    df = pd.DataFrame(linhas)
    df.to_parquet(cache, index=False)
    print(f"{vid}: {df.quadro.nunique()} quadros em {time.time() - t0:.0f} s")
    return df

DF = {vid: detectar_tudo(vid) for vid in VIDEOS}
for vid, df in DF.items():
    print(vid, df.groupby(["fonte", "classe"]).size().to_dict())

cadeirante_pexels_6646677 {('ajustado', 'pedestre'): 322, ('ajustado', 'pedestre_cadeira_rodas'): 299, ('pose', 'face'): 465, ('pose', 'person'): 489, ('yolo_world', 'bicycle'): 95, ('yolo_world', 'car'): 650, ('yolo_world', 'motorcycle'): 12, ('yolo_world', 'person'): 464, ('yolo_world', 'truck'): 1, ('yolo_world', 'wheelchair'): 163}
cadeirante_pexels_6646724 {('ajustado', 'pedestre'): 387, ('ajustado', 'pedestre_cadeira_rodas'): 257, ('pose', 'face'): 477, ('pose', 'person'): 683, ('yolo_world', 'car'): 403, ('yolo_world', 'person'): 618, ('yolo_world', 'truck'): 8, ('yolo_world', 'wheelchair'): 16}


## 2. Quem está de cadeira de rodas

In [3]:
def preparar(df):
    tid = lambda v: None if pd.isna(v) else int(v)
    yw = [vs.Deteccao(int(r.quadro), r.classe, (r.x1, r.y1, r.x2, r.y2), float(r.conf), tid(r.track_id))
          for r in df[df.fonte == "yolo_world"].itertuples()]
    aux = []
    for q, g in df[df.fonte == "ajustado"].groupby("quadro"):
        aux += vs.auxilios_do_ajustado(g[["x1", "y1", "x2", "y2"]].values.tolist(), g.classe.tolist(),
                                       g.conf.tolist(), int(q))
    extras = [vs.Deteccao(int(r.quadro), "face" if r.classe == "face" else "person", (r.x1, r.y1, r.x2, r.y2), 1.0,
                          None)
              for r in df[(df.fonte == "pose") | ((df.fonte == "ajustado") & (df.conf >= 0.1))].itertuples()]
    return yw, aux, extras

def cobertura(dets, n):
    por_q = {}
    for d in dets:
        por_q.setdefault(d.quadro, []).append(d)
    return sum(1 for ds in por_q.values() if "wheelchair" in vs.associar_auxilios(ds).values()) / n

RES, linhas = {}, []
for vid in VIDEOS:
    yw, aux, extras = preparar(DF[vid])
    so_yw = vs.sem_condutores(vs.sem_bicicleta_na_cadeira(yw))
    juntos = vs.sem_condutores(vs.sem_bicicleta_na_cadeira(yw + aux))
    n = DF[vid].quadro.nunique()
    auxilios = vs.auxilios_por_pessoa(juntos)
    RES[vid] = dict(dets=juntos, extras=extras, auxilios=auxilios)
    linhas.append({"vídeo": vid, "cena": VIDEOS[vid], "quadros": n,
                   "cadeirante etiquetado: só YOLO-World": f"{cobertura(so_yw, n):.0%}",
                   "cadeirante etiquetado: YOLO-World + ajustado": f"{cobertura(juntos, n):.0%}",
                   "ids de pessoa com auxílio": len(auxilios)})
    for t, a in auxilios.items():
        print(f"{vid}: pessoa {t} → {a} | grupo {vs.grupo_do_auxilio(a)} | ações na faixa "
              f"{sorted(vs.acoes_na_faixa(a, None))}")
tab = pd.DataFrame(linhas)
tab.to_csv(EVID / "09_cobertura_cadeirante.csv", sep=";", index=False, encoding="utf-8-sig")
tab

cadeirante_pexels_6646677: pessoa 2 → wheelchair | grupo mobilidade_reduzida | ações na faixa ['mais_tempo']


cadeirante_pexels_6646724: pessoa 2 → wheelchair | grupo mobilidade_reduzida | ações na faixa ['mais_tempo']


,vídeo,cena,quadros,cadeirante etiquetado: só YOLO-World,cadeirante etiquetado: YOLO-World + ajustado,ids de pessoa com auxílio
0,cadeirante_pexels_6646677,de frente e de lado,262,62%,87%,1
1,cadeirante_pexels_6646724,"de costas, em close",249,6%,82%,1


**Leitura:** com os dois modelos juntos, o senhor em cadeira de rodas recebe a etiqueta em **87%** dos quadros
do vídeo de frente e em **82%** do vídeo de costas. Só com o YOLO-World seriam 62% e 6%: de costas, ele quase não
reconhece a cadeira. Em cada vídeo, **uma pessoa** é marcada com cadeira de rodas, a certa. Os acompanhantes que
empurram a cadeira chegavam a "herdá-la" nos quadros em que o senhor não era detectado (14 de 234 quadros num vídeo,
61 de 249 no outro); a regra de dominância (`DOMINANCIA_PCD`) resolve isso: quem recebe a cadeira bem menos vezes do
que a pessoa ao lado é acompanhante. O YOLO-World também confundiu a cadeira com bicicleta 95 vezes; essas
"bicicletas" saem (`sem_bicicleta_na_cadeira`). Grupo de necessidade: **mobilidade reduzida → mais tempo de
travessia**.

## 3. Imagens anotadas (borradas antes, anotadas por cima)

Folha com 12 quadros (6 de cada vídeo) e GIF do vídeo de frente. Os vídeos completos, anotados e borrados, ficam em
`data/processed/` (fora do Git).

In [4]:
from PIL import Image
def quadros_anotados(vid, alvo=None):
    r = RES[vid]
    por_q, ext_q = {}, {}
    for d in r["dets"]:
        por_q.setdefault(d.quadro, []).append(d)
    for d in r["extras"]:
        ext_q.setdefault(d.quadro, []).append(d)
    _, _, _, gen = vs.ler_video(ROOT / "data" / "raw" / "video" / f"{vid}.mp4")
    for i, img in enumerate(gen):
        if alvo is None or i in alvo:
            yield i, vs.anotar(img, por_q.get(i, []), None, r["auxilios"], privadas_extra=ext_q.get(i, []))

linhas_folha = []
for vid in VIDEOS:
    alvo = set(np.linspace(5, DF[vid].quadro.max() - 10, 6).astype(int))
    linhas_folha.append(np.hstack([cv2.resize(a, (480, 270)) for _, a in quadros_anotados(vid, alvo)]))
vs.salvar_quadro(np.vstack(linhas_folha), [], EVID / "09_cadeirante_quadros.jpg")   # já borrada quadro a quadro

gif = [Image.fromarray(cv2.cvtColor(cv2.resize(a, (400, 225)), cv2.COLOR_BGR2RGB))
       for i, a in quadros_anotados("cadeirante_pexels_6646677") if i % 4 == 0]
gif[0].save(EVID / "09_cadeirante_anotado.gif", save_all=True, append_images=gif[1:], duration=167, loop=0,
            optimize=True)
print("GIF:", (EVID / "09_cadeirante_anotado.gif").stat().st_size // 1024, "KB")
for v in VIDEOS:
    fps, _, _, _ = vs.ler_video(ROOT / "data" / "raw" / "video" / f"{v}.mp4")
    saida, escritor = PROC / f"visao_{v}_anotado.mp4", None
    for _, a in quadros_anotados(v):
        if escritor is None:
            escritor = cv2.VideoWriter(str(saida), cv2.VideoWriter_fourcc(*"mp4v"), fps, (a.shape[1], a.shape[0]))
        escritor.write(a)
    escritor.release()
    print("vídeo anotado e borrado:", saida.name, f"({saida.stat().st_size / 1e6:.0f} MB)")

GIF: 3574 KB


vídeo anotado e borrado: visao_cadeirante_pexels_6646677_anotado.mp4 (14 MB)


vídeo anotado e borrado: visao_cadeirante_pexels_6646724_anotado.mp4 (12 MB)


**Leitura:** na folha, a caixa laranja com "Pedestre · cadeira de rodas" fica sobre o senhor e a
cadeira; quem empurra aparece como "Pedestre"; os carros, em azul. Todos os rostos estão pixelados, inclusive em
close e de perfil: foi aqui que se descobriu que o bloco fixo de 10 px não anonimizava rostos grandes, o que levou
ao bloco proporcional e ao rosto localizado pela pose (a correção vale para o sistema todo).

**Limitações:**
- **Só demonstração:** sem faixa e com a câmera em movimento, os vídeos não medem travessia nem conflito.
- **Dois vídeos curtos (cerca de 10 s), da mesma pessoa:** mostram que a identificação funciona, mas não são uma
  medida de desempenho geral. A medida mais justa continua sendo a das 58 fotos (notebook 07) e a do conjunto de teste
  do Roboflow (notebook 08).
- **Só cadeira de rodas** tem modelo ajustado; andador, muleta e bengala dependem do YOLO-World, que quase não os
  reconhece.
- **Limiares escolhidos olhando estes mesmos dados** (0,10 no YOLO-World, regra de dominância): os números são
  otimistas até serem confirmados em vídeos novos.